<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 4 — Admissibility Constraints: Field Content
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604845.svg)](https://doi.org/10.5281/zenodo.18604845)

**Author:** E.S. Brooke · **Paper:** v2.0 main + v1.0 supplement · **Codebase:** v6.9

---

### What this notebook is

Paper 4 is the **Standard Model field content + masses + mixings** paper. The 1-of-4680 fermion filter, $\sin^2\theta_W = 3/13$, Schur-chain mass structure, CKM, and PMNS all live here.

### What this notebook is **not**

Absolute top/bottom quark mass (Type B anchors, open), dark matter particle identity (open), or $m_c$ below 2.6% (structural Schur limit).

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content.git 2>/dev/null || true
%cd -q APF-Paper-4-Admissibility-Constraints-Field-Content
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 35 bank-registered checks bundled.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · Five-part structure

Paper 4 v2.0 (April 2026 rebuild, 39pp main + 19pp supplement) is organised as:

- **Part I** — Structural constraints under PLEC.
- **Part II** — Gauge template + 1-of-4680 fermion filter + $C_{\rm total} = 61$ + $N_{\rm gen} = 3$.
- **Part III** — $\sin^2\theta_W = 3/13$ full synthesis + masses + CKM + PMNS + DUNE.
- **Part IV** — Dark sector $C_{\rm ext}$ + $\Lambda$ as capacity residual + five falsifiers.
- **Part V** — Closure.

## §3 · The 1-of-4680 fermion filter (load-bearing)

**$T_{\rm field}$** — out of 4680 candidate $(SU(3), SU(2), U(1))$ representations, exactly ONE passes the admissibility filter:

$$\boxed{\quad \text{CP-coherence} \;+\; \text{anomaly cancellation} \;+\; \text{gauge admissibility} \;\Rightarrow\; \text{SM fermion content} \quad}$$

The 45 surviving fermions = 3 generations × 15 Weyl per generation.

In [ ]:
show('check_T_field')

## §4 · $\sin^2\theta_W = 3/13$ via gate $S_0$

Full structural derivation with zero free parameters:

$$\sin^2 \theta_W \;=\; \frac{3}{13} \;\approx\; 0.2308$$

Observed (PDG): $0.2312 \pm 0.0002$. **Agreement: 0.2σ.**

The gate-$S_0$ synthesis composes $T_{25a/b}$ + $T_{27d}$ + $T_{23}$.

In [ ]:
show('check_T_sin2theta')

## §5 · Mass structure — Schur chain

Schur matrix chain gives charm mass structurally:

$$m_c / m_t \;=\; \text{structural Schur ratio at 2.6% precision}$$

This 2.6% is the **irreducible** structural limit — the Schur chain cannot do better. See `DO_NOT_CLAIM.md` row 2.

In [ ]:
show('check_T_mass_ratios')

## §6 · CKM + PMNS + DUNE

Mixing matrices derived structurally. DUNE Δ_CP sensitivity is one of the five falsifiers.

In [ ]:
show('check_T_CKM')

In [ ]:
show('check_T_PMNS')

## §7 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18604548)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper4_2026,
  author  = {Brooke, Ethan S.},
  title   = {Admissibility Constraints and the Standard Model Field Content},
  year    = 2026,
  version = {v2.0 main + v1.0 supplement},
  doi     = {10.5281/zenodo.18604845}
}
```

*Paper-companion repo · v2.0 main + v1.0 supplement · Phase 22 gorgeous-math edition · 2026-04-24.*